In [5]:
import os

# Hugging Face 모델 저장 위치
os.environ["HF_HOME"] = "./cache/"

from langchain_teddynote import logging
from langchain_teddynote.messages import stream_response

# LangSmith 추적
logging.langsmith("CH04-Models")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH04-Models


In [6]:
%pip install -U transformers torch sentencepiece

  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.14.0-cp312-cp312-win_amd64.whl.metadata (38 kB)
  Using cached sentencepiece-0.2.2-cp312-cp312-win_amd64.whl.metadata (34 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
Using cached torch-2.14.0-cp312-cp312-win_amd64.whl (124.1 MB)
Using cached sentencepiece-0.2.2-cp312-cp312-win_amd64.whl (1.2 MB)
Using cached networkx-3.6.1-py3-none-an


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from langchain_huggingface import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="distilgpt2",
    task="text-generation",
    device=-1,
    pipeline_kwargs={
        "max_new_tokens": 50,
        "do_sample": False,
    },
)

d:\ces3356\hanwha_0902\ex0917\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\ces3356\hanwha_0902\ex0917\cache\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 570.17it/s]
[transformers] Passing `generation_config` together wit

In [13]:
response = hf.invoke(
    "What is the capital of South Korea?"
)

print(response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


What is the capital of South Korea?

The capital of South Korea was the capital of the South Korean Republic of Korea.
As a result, the capital of the South Korean Republic of Korea was the capital of South Korea.
The capital of the South Korean Republic of Korea was


In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "distilgpt2"

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 모델 로드
model = AutoModelForCausalLM.from_pretrained(model_id)

# pipeline 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    do_sample=False,
)

# LangChain용으로 감싸기
hf = HuggingFacePipeline(pipeline=pipe)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 512.52it/s]


In [18]:
response = hf.invoke(
    "Translate to Korean: I love artificial intelligence."
)

print(response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate to Korean: I love artificial intelligence. I love you. I love you.












































In [19]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template(
    "Answer the following question briefly.\n"
    "Question: {question}\n"
    "Answer:"
)

# Prompt → 로컬 모델 → 문자열
chain = prompt | hf | StrOutputParser()

In [20]:
answer = chain.invoke(
    {"question": "What is the capital of South Korea?"}
)

print(answer)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the following question briefly.
Question: What is the capital of South Korea?
Answer: The capital of South Korea is the capital of South Korea, but not in the South Korean capital. However, it has no government. The only government in South Korea is the National Party of Korea. The National Party of Korea is the Party of Korea
